In [ ]:
import torch
from torch import nn

In [2]:
from torch import Tensor
from typing import Literal
class MaskedCNN(nn.Conv2d):
    def __init__(self, type: Literal["A", "B"] = "A",  *args, **kwargs) -> None:
        super(MaskedCNN, self).__init__(*args, **kwargs)
        out_channels, in_channels, height, width = self.weight.shape
        self.register_buffer("mask", torch.ones_like(self.weight))
        
        if type == "A":
            self.mask[:, :, height//2:, width // 2:] = 0
            self.mask[: , :, height // 2 + 1:, :] = 0
        else:
            self.mask[:, :, height//2:, width // 2 + 1:] = 0
            self.mask[: , :, height // 2 + 1:, :] = 0
    
        
        
    def forward(self, x: Tensor) -> Tensor:
        return super(MaskedCNN, self).forward(x)
        
        
        

In [17]:
from typing import Any

from torch._functorch.vmap import _num_outputs


class PixelCNN(nn.Module):
    def __init__(
        self,
        in_channels: int,
        channels: int,
        num_layers: int,
        kernel_size: int,
        bit_depth: int,
    ) -> None:
        super().__init__()

        self.num_outputs = 2**bit_depth

        layers = [
            MaskedCNN(
                type="A",
                in_channels=in_channels,
                out_channels=channels,
                kernel_size=kernel_size,
                stride=1,
                padding="same",
            ),
            nn.BatchNorm2d(num_features=channels),
            nn.ReLU(),
        ]

        for i in range(num_layers - 1):
            layers.extend(
                [
                    MaskedCNN(
                        type="B",
                        in_channels=channels,
                        out_channels=channels,
                        kernel_size=kernel_size,
                        stride=1,
                        padding="same",
                    ),
                    nn.BatchNorm2d(num_features=channels),
                    nn.ReLU(),
                ]
            )
        print(f"LEN layers: {len(layers)}")
        self.layers = nn.ModuleList(layers)
        self.output_proj = nn.Conv2d(
            in_channels=channels,
            out_channels=in_channels * self.num_outputs,
            kernel_size=1,
        )

    def forward(self, x: Tensor):
        batch, ch, h, w = x.shape
        for layer in self.layers:
            x = layer(x)

        x = self.output_proj(x)
        return x.reshape(batch, ch, self.num_outputs, h, w).permute(0, 2, 1, 3, 4)

In [18]:
model = PixelCNN(
    1, 64, 8, 7, 8
)
def generate_samples():
    pass

LEN layers: 24


In [5]:
from torchvision import datasets
from torch.utils.data import DataLoader
import torchvision
mnist = datasets.MNIST(root="data", train=True, transform=torchvision.transforms.ToTensor(), download= True)
loader = DataLoader(mnist)

100%|██████████| 9.91M/9.91M [00:10<00:00, 923kB/s] 
100%|██████████| 28.9k/28.9k [00:00<00:00, 148kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.17MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.87MB/s]


In [ ]:
from torch.optim import Adam
optimizer = Adam(params = model.parameters(), lr = 1e-3)
loss_fn = nn.CrossEntropyLoss()
device='cpu'
from tqdm import tqdm
epochs =10
for i in range(epochs):
    model.train()
    print(f"Epoch {i}")
    pbar = tqdm(loader, desc=f"Epoch {i}")
    for idx, (x, _) in enumerate( pbar):   
        with torch.autocast(device_type=device, dtype=torch.bfloat16, enabled=True):
            logits = model(x)
            targets = (x * 255).to(torch.long)
            loss = loss_fn(logits, targets)
        # pbar.update(
            
        # )
        loss.backward()
        optimizer.step()
        



In [24]:
def generate_one(num_samples: int = 16, image_size =  28, num_channels = 1,device="cpu"):
    model.eval()
    samples = torch.zeros(
        num_samples, num_channels, image_size, dtype=torch.long
    )
    with torch.no_grad():
        for i in range(image_size):
            for j in range(image_size):
                output = model(samples)
                for ch in range( num_channels):
                    logit_ch =  output[:, :, ch, i, j]
                    probs = torch.softmax(logit_ch, dim=1)
                    samples[:, ch, i, j]= torch.multinomial(probs, 1).squeeze(-1)
                    
    model.train()
    return samples
    

#### Code is copied from https://www.youtube.com/watch?v=9h2NekDz4ag